In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Retrieve widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Determine file format (default to parquet if not csv)
_file_format = source_file_format.lower()
_landing_path = f"/Volumes/ws_adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}"

if _file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(_landing_path)
else:
    landing_df = spark.read.format("parquet").load(_landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp, to_timestamp

# Add landing_timestamp (from widget) and insert_timestamp (current time)
updated_df = (
    landing_df
    .withColumn("landing_timestamp", to_timestamp(lit(landing_timestamp), 'yyyy-MM-dd_HH:mm:ss'))
    .withColumn("insert_timestamp", current_timestamp())
)


In [0]:
updated_df.write.mode("append").saveAsTable(f"ws_adb_caresync.bronze.{table_name}")